<a href="https://colab.research.google.com/github/antoniovfonseca/agentic-ai-global-lulc/blob/main/MCP_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environment Setup

Install the official `mcp` (Model Context Protocol) SDK and the `numpy` library for numerical processing.

In [2]:
# 1. Install the Model Context Protocol SDK
# 2. Install NumPy library for mathematical functions
!pip install -q mcp numpy

## 2. Imports and Server Initialization

Import the libraries and initialize the FastMCP server instance.

In [3]:
import numpy as np
from mcp.server.fastmcp import FastMCP

# 1. Initialize the FastMCP server with a specific name
# 2. This instance will handle tool registration and requests
mcp = FastMCP("My Colab MCP")

## 3. Tool Registration

Define a function decorated with `@mcp.tool()`. This example uses `numpy` to calculate statistics (mean and standard deviation) from a list of numbers, demonstrating how to expose Python logic to the MCP server.

In [4]:
@mcp.tool()
def calculate_statistics(data: list[float]) -> str:
    """Calculates mean and standard deviation of a dataset."""

    # 1. Convert the input list to a NumPy array for processing
    array_data = np.array(data)

    # 2. Calculate the mean value of the data
    mean_val = np.mean(array_data)

    # 3. Calculate the standard deviation of the data
    std_val = np.std(array_data)

    # 4. Format and return the results as a string
    return f"Mean: {mean_val:.2f}, Standard Deviation: {std_val:.2f}"

## 4. Local Tool Verification

Test the registered tool by invoking the function directly with sample data. This ensures the logic behaves as expected before starting the server process.

In [5]:
# 1. Define a sample list of floats for testing
sample_data = [10.5, 22.3, 33.1, 45.0, 12.8]

# 2. Call the function directly to verify the output
result = calculate_statistics(sample_data)

# 3. Print the formatted result to the console
print(f"Test Result: {result}")

Test Result: Mean: 24.74, Standard Deviation: 12.90


## 5. Running the Server (Threaded Approach)

Since Google Colab controls the main event loop, I execute the MCP server in a separate thread. This prevents the `asyncio` conflict and allows the server to run in the background.

In [9]:
import threading

# 1. Define a wrapper function to run the server logic
def run_mcp_server():
    try:
        # mcp.run() creates its own event loop, which works fine inside a new thread
        mcp.run()
    except Exception as e:
        print(f"\nServer error: {e}")

# 2. Create a new thread targeting the wrapper function
# daemon=True ensures the thread closes when the notebook kernel shuts down
server_thread = threading.Thread(target=run_mcp_server, daemon=True)

# 3. Start the server thread
print("Starting MCP Server in a background thread...")
server_thread.start()

# 4. Confirmation message
print("Server is running! You can now continue using other cells.")

Starting MCP Server in a background thread...
Server is running! You can now continue using other cells.


## 6. Template for Custom Tools

To add new capabilities, define functions with the `@mcp.tool()` decorator **before** running the server loop (Step 5).

**Checklist for new tools:**
1.  **Type Hints:** Always specify input types (e.g., `x: int`) and return types.
2.  **Docstrings:** Write clear descriptions. The LLM reads this to understand when to use the tool.
3.  **Return Value:** Return a string or a JSON-serializable object.

In [11]:
# --- TEMPLATE FOR NEW TOOLS ---

# @mcp.tool()
# def my_new_tool(input_text: str, repeat_count: int) -> str:
#     """
#     Description: Explain what this tool does clearly.
#     Args:
#         input_text: Description of the first argument.
#         repeat_count: Description of the second argument.
#     """
#     # 1. Your logic here (e.g., data processing, API calls)
#     result = input_text * repeat_count
#
#     # 2. Return the output
#     return f"Processed: {result}"